# Сравнение моделей: 17 vs 19 vs 21 фич

- **17 фич** — базовая модель (синтетика без нормативов)
- **19 фич** — + `grazing_norm_deviation`, `natural_loss_risk_score` (гос. нормативы)
- **21 фич** — + `breeding_ratio_compliance`, `application_success_rate` (эксперимент)

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("../data")
MODELS_DIR = Path("../models")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(exist_ok=True)

## 1. Загрузка датасетов

In [ ]:
df_19 = pd.read_csv(DATA_DIR / "data_features.csv")
df_21 = pd.read_csv(DATA_DIR / "data_features_21.csv")

print(f"19 фич: {len(df_19):,} строк, {df_19.shape[1]} колонок")
print(f"21 фич: {len(df_21):,} строк, {df_21.shape[1]} колонок")
print(f"\n19 фич: {list(df_19.columns)}")
print(f"\n21 фич: {list(df_21.columns)}")

## 2. Наборы фич

In [ ]:
FEATURES_17 = [
    "gross_output_growth_yoy", "land_to_livestock_ratio",
    "historical_survival_rate", "subsidy_dependence_index",
    "veterinary_compliance", "years_in_operation",
    "pedigree_ratio", "previous_subsidies_count", "debt_load_ratio",
    "log_amount", "livestock_count", "direction_code",
    "is_pedigree", "is_producer", "hour_submitted",
    "month_submitted", "region_encoded",
]

FEATURES_19 = FEATURES_17 + ["grazing_norm_deviation", "natural_loss_risk_score"]

FEATURES_21 = FEATURES_19 + ["breeding_ratio_compliance", "application_success_rate"]

TARGET = "historical_score"

print(f"17 фич: {len(FEATURES_17)}")
print(f"19 фич: {len(FEATURES_19)} (+{len(FEATURES_19) - len(FEATURES_17)})")
print(f"21 фич: {len(FEATURES_21)} (+{len(FEATURES_21) - len(FEATURES_19)})")

## 3. Общий train/test split

In [ ]:
# Используем 19-фич датасет как основу (он уже с нормативами)
X_17 = df_19[FEATURES_17].copy()
X_19 = df_19[FEATURES_19].copy()
y = df_19[TARGET].copy()

# 21-фич датасет — тот же индекс
X_21 = df_21[FEATURES_21].copy()

# Очистка
mask = X_17.notna().all(axis=1) & y.notna()
X_17 = X_17[mask]
y = y[mask]
X_19 = X_19.loc[mask]
X_21 = X_21.loc[mask]

# Split
X_17_train, X_17_test, y_train, y_test = train_test_split(
    X_17, y, test_size=0.20, random_state=RANDOM_SEED
)
X_19_train = X_19.loc[X_17_train.index]
X_19_test = X_19.loc[X_17_test.index]
X_21_train = X_21.loc[X_17_train.index]
X_21_test = X_21.loc[X_17_test.index]

print(f"Train: {len(X_17_train):,} строк")
print(f"Test:  {len(X_17_test):,} строк")

## 4. Функция обучения и оценки

In [ ]:
def build_model():
    return XGBRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7,
        reg_lambda=1.0, reg_alpha=0.1, min_child_weight=5,
        early_stopping_rounds=50, random_state=RANDOM_SEED,
        n_jobs=-1, verbosity=0,
    )

def train_and_evaluate(X_train, X_test, y_train, y_test, label: str) -> dict:
    print(f"\n{'='*60}")
    print(f"  {label} ({X_train.shape[1]} фич)")
    print(f"{'='*60}")
    
    scaler = StandardScaler()
    X_tr = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_te = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
    
    model = build_model()
    model.fit(X_tr, y_train, eval_set=[(X_te, y_test)], verbose=50)
    
    y_pred_tr = np.clip(model.predict(X_tr), 1, 100)
    y_pred_te = np.clip(model.predict(X_te), 1, 100)
    
    metrics = {
        "label": label,
        "n_features": X_train.shape[1],
        "n_trees": model.best_iteration + 1,
        "train_mae":  round(mean_absolute_error(y_train, y_pred_tr), 3),
        "train_rmse": round(np.sqrt(mean_squared_error(y_train, y_pred_tr)), 3),
        "train_r2":   round(r2_score(y_train, y_pred_tr), 4),
        "test_mae":   round(mean_absolute_error(y_test, y_pred_te), 3),
        "test_rmse":  round(np.sqrt(mean_squared_error(y_test, y_pred_te)), 3),
        "test_r2":    round(r2_score(y_test, y_pred_te), 4),
    }
    
    print(f"  5-fold CV...")
    cv_model = XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        random_state=RANDOM_SEED, n_jobs=-1, verbosity=0,
    )
    cv_scores = cross_val_score(cv_model, X_tr, y_train, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
    metrics["cv_mae"] = round(-cv_scores.mean(), 3)
    metrics["cv_std"] = round(cv_scores.std(), 3)
    
    df_err = pd.DataFrame({"y_true": y_test, "y_pred": y_pred_te})
    df_err["zone"] = pd.cut(df_err["y_true"], bins=[0, 50, 80, 100], labels=["red", "yellow", "green"])
    zone_mae = df_err.groupby("zone", observed=True).apply(
        lambda g: round(mean_absolute_error(g["y_true"], g["y_pred"]), 2)
    ).to_dict()
    metrics["mae_by_zone"] = zone_mae
    
    importances = pd.Series(model.feature_importances_, index=X_train.columns)
    metrics["feature_importance"] = importances.sort_values(ascending=False)
    
    print(f"  Деревьев: {metrics['n_trees']}")
    print(f"  Train MAE:  {metrics['train_mae']:.3f}  |  RMSE: {metrics['train_rmse']:.3f}  |  R²: {metrics['train_r2']:.4f}")
    print(f"  Test  MAE:  {metrics['test_mae']:.3f}  |  RMSE: {metrics['test_rmse']:.3f}  |  R²: {metrics['test_r2']:.4f}")
    print(f"  CV MAE:     {metrics['cv_mae']:.3f} ± {metrics['cv_std']:.3f}")
    print(f"  MAE по зонам: {zone_mae}")
    
    return metrics, model, scaler

## 5. Обучение всех трёх моделей

In [ ]:
print("🚀 МОДЕЛЬ 1: 17 фич (базовая)...")
m17, model_17, scaler_17 = train_and_evaluate(
    X_17_train, X_17_test, y_train, y_test, "17 фич (базовая)"
)

In [ ]:
print("🚀 МОДЕЛЬ 2: 19 фич (+нормативы)...")
m19, model_19, scaler_19 = train_and_evaluate(
    X_19_train, X_19_test, y_train, y_test, "19 фич (+нормативы)"
)

In [ ]:
print("🚀 МОДЕЛЬ 3: 21 фич (+эксперимент)...")
m21, model_21, scaler_21 = train_and_evaluate(
    X_21_train, X_21_test, y_train, y_test, "21 фич (+эксперимент)"
)

## 6. Сравнительная таблица

In [ ]:
def build_comparison():
    rows = []
    for key in ["train_mae", "train_rmse", "train_r2", "test_mae", "test_rmse", "test_r2", "cv_mae"]:
        vals = {"17": m17[key], "19": m19[key], "21": m21[key]}
        delta_17_19 = vals["19"] - vals["17"]
        delta_19_21 = vals["21"] - vals["19"]
        delta_17_21 = vals["21"] - vals["17"]
        
        is_lower_better = "r2" not in key
        
        def arrow(delta, better_lower):
            if better_lower:
                return "✅" if delta < 0 else "❌"
            return "✅" if delta > 0 else "❌"
        
        rows.append({
            "Метрика": key,
            "17 фич": vals["17"],
            "19 фич": vals["19"],
            "Δ 17→19": f"{delta_17_19:+.4f} {arrow(delta_17_19, is_lower_better)}",
            "21 фич": vals["21"],
            "Δ 19→21": f"{delta_19_21:+.4f} {arrow(delta_19_21, is_lower_better)}",
            "Δ 17→21": f"{delta_17_21:+.4f} {arrow(delta_17_21, is_lower_better)}",
        })
    
    for zone in ["red", "yellow", "green"]:
        vals = {
            "17": m17["mae_by_zone"].get(zone),
            "19": m19["mae_by_zone"].get(zone),
            "21": m21["mae_by_zone"].get(zone),
        }
        if all(v is not None for v in vals.values()):
            d1 = vals["19"] - vals["17"]
            d2 = vals["21"] - vals["19"]
            d3 = vals["21"] - vals["17"]
            rows.append({
                "Метрика": f"MAE зона {zone}",
                "17 фич": vals["17"],
                "19 фич": vals["19"],
                "Δ 17→19": f"{d1:+.4f} {'✅' if d1 < 0 else '❌'}",
                "21 фич": vals["21"],
                "Δ 19→21": f"{d2:+.4f} {'✅' if d2 < 0 else '❌'}",
                "Δ 17→21": f"{d3:+.4f} {'✅' if d3 < 0 else '❌'}",
            })
    
    return pd.DataFrame(rows)

comparison = build_comparison()
display(comparison)

## 7. Визуализация

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
labels = ["17 фич", "19 фич", "21 фич"]
colors = ["#1976d2", "#4caf50", "#ff6f00"]
x = np.arange(3)
w = 0.5

# MAE
mae_vals = [m17["test_mae"], m19["test_mae"], m21["test_mae"]]
bars = axes[0].bar(x, mae_vals, w, color=colors, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels)
axes[0].set_ylabel("Test MAE (меньше = лучше)")
axes[0].set_title("Test MAE")
for i, v in enumerate(mae_vals):
    axes[0].text(i, v + 0.03, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")

# R²
r2_vals = [m17["test_r2"], m19["test_r2"], m21["test_r2"]]
bars = axes[1].bar(x, r2_vals, w, color=colors, alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_ylabel("Test R² (больше = лучше)")
axes[1].set_title("Test R²")
for i, v in enumerate(r2_vals):
    axes[1].text(i, v + 0.002, f"{v:.4f}", ha="center", fontsize=11, fontweight="bold")

# CV MAE
cv_vals = [m17["cv_mae"], m19["cv_mae"], m21["cv_mae"]]
bars = axes[2].bar(x, cv_vals, w, color=colors, alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels)
axes[2].set_ylabel("CV MAE (меньше = лучше)")
axes[2].set_title("5-Fold CV MAE")
for i, v in enumerate(cv_vals):
    axes[2].text(i, v + 0.03, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(REPORTS_DIR / "compare_17_19_21.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"💾 {REPORTS_DIR / 'compare_17_19_21.png'}")

## 8. Feature Importance — все 21 фича

In [ ]:
fi = m21["feature_importance"].copy()
fi_norm = fi / fi.sum() * 100

fig, ax = plt.subplots(figsize=(10, 8))

colors_fi = []
for feat in fi_norm.index:
    if feat in ["breeding_ratio_compliance", "application_success_rate"]:
        colors_fi.append("#ff6f00")  # эксперимент
    elif feat in ["grazing_norm_deviation", "natural_loss_risk_score"]:
        colors_fi.append("#4caf50")  # нормативы
    elif fi_norm[feat] < fi_norm.median():
        colors_fi.append("#d32f2f")
    else:
        colors_fi.append("#1976d2")

fi_norm.plot(kind="barh", ax=ax, color=colors_fi)
ax.set_title("Feature Importance — 21 фича", fontsize=14, fontweight="bold")
ax.set_xlabel("Доля важности (%)")
ax.axvline(fi_norm.median(), color="orange", linestyle="--", alpha=0.7, label="Медиана")
ax.legend()

for feat in ["breeding_ratio_compliance", "application_success_rate",
             "grazing_norm_deviation", "natural_loss_risk_score"]:
    val = fi_norm[feat]
    idx = list(fi_norm.index).index(feat)
    ax.text(val + 0.2, idx, f"{val:.1f}%", va="center", fontsize=10,
            color="#ff6f00" if "breeding" in feat or "application" in feat else "#4caf50",
            fontweight="bold")

plt.tight_layout()
plt.savefig(REPORTS_DIR / "feature_importance_21.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📊 Важность новых фич:")
for feat in ["grazing_norm_deviation", "natural_loss_risk_score",
             "breeding_ratio_compliance", "application_success_rate"]:
    print(f"  {feat}: {fi_norm[feat]:.2f}%")

## 9. Итоговый вывод

In [ ]:
print("=" * 70)
print("  ИТОГОВОЕ СРАВНЕНИЕ: 17 vs 19 vs 21 фич")
print("=" * 70)

for label, m in [("17 фич", m17), ("19 фич", m19), ("21 фич", m21)]:
    print(f"\n  {label}:")
    print(f"    Test MAE: {m['test_mae']:.3f}  |  R²: {m['test_r2']:.4f}  |  CV MAE: {m['cv_mae']:.3f}")

print(f"\n  Улучшение 17→19: MAE {m17['test_mae']:.3f} → {m19['test_mae']:.3f} ({(m19['test_mae']-m17['test_mae'])/m17['test_mae']*100:+.1f}%)")
print(f"  Улучшение 19→21: MAE {m19['test_mae']:.3f} → {m21['test_mae']:.3f} ({(m21['test_mae']-m19['test_mae'])/m19['test_mae']*100:+.1f}%)")
print(f"  Улучшение 17→21: MAE {m17['test_mae']:.3f} → {m21['test_mae']:.3f} ({(m21['test_mae']-m17['test_mae'])/m17['test_mae']*100:+.1f}%)")

best_mae = min(m17["test_mae"], m19["test_mae"], m21["test_mae"])
best_label = {m17["test_mae"]: "17", m19["test_mae"]: "19", m21["test_mae"]: "21"}[best_mae]

print(f"\n  🏆 Лучшая по MAE: {best_label} фич ({best_mae:.3f})")

if best_label == "21":
    print("  ✅ Экспериментальные фичи УЛУЧШИЛИ модель!")
elif best_label == "19":
    print("  ⚠️ Экспериментальные фичи НЕ дали улучшения — оставить 19")
else:
    print("  ❌ Даже нормативы не помогли — проблема в данных")

print("=" * 70)